In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("../data/raw")

In [2]:
sales = pd.read_csv(DATA_DIR / "sales_daily.csv")
inventory = pd.read_csv(DATA_DIR / "inventory_snapshots.csv")
calendar = pd.read_csv(DATA_DIR / "calendar.csv")
sku = pd.read_csv(DATA_DIR / "sku_master.csv")

In [3]:
datasets = {
    "Sales": sales,
    "Inventory": inventory,
    "Calendar": calendar,
    "SKU Master": sku
}

for name, df in datasets.items():
    print(f"\n{'='*50}")
    print(name)
    print(f"{'='*50}")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print("Duplicate rows:", df.duplicated().sum())
    print("Missing values:", df.isna().sum().sum())


Sales
Shape: (36550, 6)
Columns: ['Date', 'SKU', 'Units_Sold', 'Revenue', 'Price', 'Promotion']
Duplicate rows: 0
Missing values: 0

Inventory
Shape: (4800, 8)
Columns: ['Snapshot_Date', 'SKU', 'Current_Stock', 'On_Order', 'Lead_Time_Days', 'Safety_Stock', 'Reorder_Point', 'Inventory_Value']
Duplicate rows: 0
Missing values: 0

Calendar
Shape: (731, 11)
Columns: ['date', 'year', 'month', 'quarter', 'week', 'day_of_week', 'is_weekend', 'season', 'holiday', 'is_holiday', 'promotion_event']
Duplicate rows: 0
Missing values: 1379

SKU Master
Shape: (50, 8)
Columns: ['SKU', 'Product_Name', 'Category', 'Subcategory', 'Launch_Date', 'Cost_Price', 'Selling_Price', 'Gross_Margin_Per_Unit']
Duplicate rows: 0
Missing values: 0


In [4]:
display(sales.head())
display(inventory.head())
display(calendar.head())
display(sku.head())

,Date,SKU,Units_Sold,Revenue,Price,Promotion
0,2024-01-01,SKU001,5,18320.25,3664.05,0
1,2024-01-01,SKU002,15,57085.35,3805.69,0
2,2024-01-01,SKU003,5,40391.15,8078.23,0
3,2024-01-01,SKU004,5,34307.85,6861.57,0
4,2024-01-01,SKU005,12,113918.64,9493.22,0


,Snapshot_Date,SKU,Current_Stock,On_Order,Lead_Time_Days,Safety_Stock,Reorder_Point,Inventory_Value
0,2024-01-01,SKU001,16,23,11,7,18,58420.96
1,2024-01-01,SKU002,29,12,4,5,10,163983.40
2,2024-01-01,SKU003,34,23,14,6,20,209060.22
3,2024-01-01,SKU004,34,4,7,5,12,42257.92
4,2024-01-01,SKU005,23,5,5,6,11,170945.89


,date,year,month,quarter,week,day_of_week,is_weekend,season,holiday,is_holiday,promotion_event
0,2024-01-01,2024,1,Q1,1,Monday,0,Winter,NaN,0,NaN
1,2024-01-02,2024,1,Q1,1,Tuesday,0,Winter,NaN,0,NaN
2,2024-01-03,2024,1,Q1,1,Wednesday,0,Winter,NaN,0,NaN
3,2024-01-04,2024,1,Q1,1,Thursday,0,Winter,NaN,0,NaN
4,2024-01-05,2024,1,Q1,1,Friday,0,Winter,NaN,0,NaN


,SKU,Product_Name,Category,Subcategory,Launch_Date,Cost_Price,Selling_Price,Gross_Margin_Per_Unit
0,SKU001,Product 001,Furniture,Chair,2022-04-09,1758.45,3664.05,1905.60
1,SKU002,Product 002,Home Decor,Table,2024-05-01,3867.09,3805.69,-61.40
2,SKU003,Product 003,Kitchen,Cushion,2023-12-22,589.48,8078.23,7488.75
3,SKU004,Product 004,Lighting,Cookware,2023-04-28,1445.74,6861.57,5415.83
4,SKU005,Product 005,Storage,Lamp,2023-04-22,5543.63,9493.22,3949.59


In [5]:
# STEP 2 — Data Quality & Coverage Checks

# --------------------------------------------------
# 1. Convert date columns to datetime
# --------------------------------------------------

sales["Date"] = pd.to_datetime(sales["Date"])
inventory["Snapshot_Date"] = pd.to_datetime(inventory["Snapshot_Date"])
calendar["date"] = pd.to_datetime(calendar["date"])
sku["Launch_Date"] = pd.to_datetime(sku["Launch_Date"])


# --------------------------------------------------
# 2. Date ranges
# --------------------------------------------------

print("DATE RANGES")
print("-" * 50)

print("Sales:     ", sales["Date"].min().date(), "to", sales["Date"].max().date())
print("Inventory: ", inventory["Snapshot_Date"].min().date(), "to", inventory["Snapshot_Date"].max().date())
print("Calendar:  ", calendar["date"].min().date(), "to", calendar["date"].max().date())
print("SKU Master launch dates:",
      sku["Launch_Date"].min().date(), "to", sku["Launch_Date"].max().date())



print("Sales:     ", sales["Date"].min().date(), "to", sales["Date"].max().date())


# --------------------------------------------------
# 3. Unique SKU counts
# --------------------------------------------------

print("\nUNIQUE SKUs")
print("-" * 50)

print("Sales SKUs:     ", sales["SKU"].nunique())
print("Inventory SKUs: ", inventory["SKU"].nunique())
print("SKU Master:     ", sku["SKU"].nunique())


# --------------------------------------------------
# 4. Check whether SKUs match across datasets
# --------------------------------------------------

sales_skus = set(sales["SKU"])
inventory_skus = set(inventory["SKU"])
master_skus = set(sku["SKU"])

print("\nSKU CONSISTENCY")
print("-" * 50)

print("Sales vs Master - missing from master:",
      sales_skus - master_skus)

print("Inventory vs Master - missing from master:",
      inventory_skus - master_skus)

print("Master vs Sales - not found in sales:",
      master_skus - sales_skus)

print("Master vs Inventory - not found in inventory:",
      master_skus - inventory_skus)


# --------------------------------------------------
# 5. Duplicate business keys
# --------------------------------------------------

print("\nDUPLICATE BUSINESS KEYS")
print("-" * 50)

print(
    "Sales duplicate Date + SKU:",
    sales.duplicated(subset=["Date", "SKU"]).sum()
)

print(
    "Inventory duplicate Snapshot_Date + SKU:",
    inventory.duplicated(subset=["Snapshot_Date", "SKU"]).sum()
)

print(
    "Calendar duplicate date:",
    calendar.duplicated(subset=["date"]).sum()
)

print(
    "SKU Master duplicate SKU:",
    sku.duplicated(subset=["SKU"]).sum()
)


# --------------------------------------------------
# 6. Numeric sanity checks
# --------------------------------------------------

print("\nNUMERIC SANITY CHECKS")
print("-" * 50)

print("Sales:")
print("  Negative Units Sold:", (sales["Units_Sold"] < 0).sum())
print("  Zero Units Sold:    ", (sales["Units_Sold"] == 0).sum())
print("  Negative Revenue:   ", (sales["Revenue"] < 0).sum())
print("  Negative Price:     ", (sales["Price"] < 0).sum())

print("\nInventory:")
print("  Negative Current Stock:",
      (inventory["Current_Stock"] < 0).sum())
print("  Negative On Order:",
      (inventory["On_Order"] < 0).sum())
print("  Negative Lead Time:",
      (inventory["Lead_Time_Days"] < 0).sum())
print("  Negative Safety Stock:",
      (inventory["Safety_Stock"] < 0).sum())
print("  Negative Reorder Point:",
      (inventory["Reorder_Point"] < 0).sum())

print("\nSKU Master:")
print("  Negative Cost Price:",
      (sku["Cost_Price"] < 0).sum())
print("  Negative Selling Price:",
      (sku["Selling_Price"] < 0).sum())
print("  Negative Gross Margin:",
      (sku["Gross_Margin_Per_Unit"] < 0).sum())


# --------------------------------------------------
# 7. Calendar missing values by column
# --------------------------------------------------

print("\nCALENDAR MISSING VALUES")
print("-" * 50)

print(calendar.isna().sum())

DATE RANGES
--------------------------------------------------
Sales:      2024-01-01 to 2025-12-31
Inventory:  2024-01-01 to 2025-12-01
Calendar:   2024-01-01 to 2025-12-31
SKU Master launch dates: 2022-03-12 to 2024-12-09
Sales:      2024-01-01 to 2025-12-31

UNIQUE SKUs
--------------------------------------------------
Sales SKUs:      50
Inventory SKUs:  200
SKU Master:      50

SKU CONSISTENCY
--------------------------------------------------
Sales vs Master - missing from master: set()
Inventory vs Master - missing from master: {'SKU098', 'SKU064', 'SKU061', 'SKU060', 'SKU127', 'SKU139', 'SKU185', 'SKU100', 'SKU131', 'SKU120', 'SKU078', 'SKU076', 'SKU172', 'SKU132', 'SKU077', 'SKU057', 'SKU059', 'SKU063', 'SKU178', 'SKU101', 'SKU166', 'SKU095', 'SKU189', 'SKU105', 'SKU052', 'SKU119', 'SKU151', 'SKU146', 'SKU187', 'SKU193', 'SKU093', 'SKU134', 'SKU080', 'SKU181', 'SKU054', 'SKU173', 'SKU068', 'SKU055', 'SKU200', 'SKU130', 'SKU090', 'SKU065', 'SKU087', 'SKU069', 'SKU096', 'SKU109

In [6]:
# STEP 3 — Investigate Inventory SKU Mismatch

# SKUs that belong to the FORESIGHT project
project_skus = set(sku["SKU"])

# SKUs appearing only in inventory
extra_inventory_skus = set(inventory["SKU"]) - project_skus

print("Total inventory SKUs:", inventory["SKU"].nunique())
print("Project SKUs:", len(project_skus))
print("Extra inventory SKUs:", len(extra_inventory_skus))

print("\nFirst 20 extra inventory SKUs:")
print(sorted(extra_inventory_skus)[:20])


# Compare number of inventory rows
project_inventory = inventory[inventory["SKU"].isin(project_skus)]
extra_inventory = inventory[~inventory["SKU"].isin(project_skus)]

print("\nROW COUNTS")
print("-" * 40)
print("Total inventory rows:", len(inventory))
print("Project SKU inventory rows:", len(project_inventory))
print("Extra SKU inventory rows:", len(extra_inventory))


# Snapshot dates
print("\nSNAPSHOT DATES")
print("-" * 40)
print("Total snapshot dates:", inventory["Snapshot_Date"].nunique())
print("Project SKU snapshot dates:", project_inventory["Snapshot_Date"].nunique())
print("Extra SKU snapshot dates:", extra_inventory["Snapshot_Date"].nunique())


# Numeric sanity checks
print("\nNUMERIC SANITY CHECKS")
print("-" * 40)

print("Negative Units Sold:",
      (sales["Units_Sold"] < 0).sum())

print("Negative Revenue:",
      (sales["Revenue"] < 0).sum())

print("Negative Price:",
      (sales["Price"] < 0).sum())

print("Negative Current Stock:",
      (inventory["Current_Stock"] < 0).sum())

print("Negative On Order:",
      (inventory["On_Order"] < 0).sum())

print("Negative Lead Time:",
      (inventory["Lead_Time_Days"] < 0).sum())

print("Negative Safety Stock:",
      (inventory["Safety_Stock"] < 0).sum())

print("Negative Reorder Point:",
      (inventory["Reorder_Point"] < 0).sum())

print("Negative Inventory Value:",
      (inventory["Inventory_Value"] < 0).sum())

print("\nNegative Gross Margin SKUs:",
      (sku["Gross_Margin_Per_Unit"] < 0).sum())

Total inventory SKUs: 200
Project SKUs: 50
Extra inventory SKUs: 150

First 20 extra inventory SKUs:
['SKU051', 'SKU052', 'SKU053', 'SKU054', 'SKU055', 'SKU056', 'SKU057', 'SKU058', 'SKU059', 'SKU060', 'SKU061', 'SKU062', 'SKU063', 'SKU064', 'SKU065', 'SKU066', 'SKU067', 'SKU068', 'SKU069', 'SKU070']

ROW COUNTS
----------------------------------------
Total inventory rows: 4800
Project SKU inventory rows: 1200
Extra SKU inventory rows: 3600

SNAPSHOT DATES
----------------------------------------
Total snapshot dates: 24
Project SKU snapshot dates: 24
Extra SKU snapshot dates: 24

NUMERIC SANITY CHECKS
----------------------------------------
Negative Units Sold: 0
Negative Revenue: 0
Negative Price: 0
Negative Current Stock: 0
Negative On Order: 0
Negative Lead Time: 0
Negative Safety Stock: 0
Negative Reorder Point: 0
Negative Inventory Value: 0

Negative Gross Margin SKUs: 16


In [7]:
# STEP 4 — Create Project Inventory + Inspect Calendar

# --------------------------------------------------
# 1. Keep only the 50 SKUs in our project universe
# --------------------------------------------------

project_skus = set(sku["SKU"])

project_inventory = inventory[
    inventory["SKU"].isin(project_skus)
].copy()

print("PROJECT INVENTORY")
print("-" * 50)
print("Rows:", project_inventory.shape[0])
print("Unique SKUs:", project_inventory["SKU"].nunique())
print("Snapshot dates:", project_inventory["Snapshot_Date"].nunique())


# --------------------------------------------------
# 2. Check the actual inventory snapshot dates
# --------------------------------------------------

print("\nINVENTORY SNAPSHOT DATES")
print("-" * 50)

print(
    project_inventory["Snapshot_Date"]
    .drop_duplicates()
    .sort_values()
    .dt.strftime("%Y-%m-%d")
    .tolist()
)


# --------------------------------------------------
# 3. Inspect calendar categorical values
# --------------------------------------------------

print("\nCALENDAR VALUES")
print("-" * 50)

print("Holiday values:")
print(calendar["holiday"].value_counts(dropna=False))

print("\nPromotion event values:")
print(calendar["promotion_event"].value_counts(dropna=False))


# --------------------------------------------------
# 4. Check whether missing holiday/promotion means
#    'nothing happening'
# --------------------------------------------------

print("\nCALENDAR EVENT COUNTS")
print("-" * 50)

print("Holiday populated:",
      calendar["holiday"].notna().sum())

print("Holiday missing:",
      calendar["holiday"].isna().sum())

print("Promotion populated:",
      calendar["promotion_event"].notna().sum())

print("Promotion missing:",
      calendar["promotion_event"].isna().sum())


# --------------------------------------------------
# 5. List negative-margin products
# --------------------------------------------------

negative_margin_skus = sku[
    sku["Gross_Margin_Per_Unit"] < 0
][[
    "SKU",
    "Product_Name",
    "Category",
    "Cost_Price",
    "Selling_Price",
    "Gross_Margin_Per_Unit"
]]

print("\nNEGATIVE MARGIN PRODUCTS")
print("-" * 50)

display(negative_margin_skus)

PROJECT INVENTORY
--------------------------------------------------
Rows: 1200
Unique SKUs: 50
Snapshot dates: 24

INVENTORY SNAPSHOT DATES
--------------------------------------------------
['2024-01-01', '2024-02-01', '2024-03-01', '2024-04-01', '2024-05-01', '2024-06-01', '2024-07-01', '2024-08-01', '2024-09-01', '2024-10-01', '2024-11-01', '2024-12-01', '2025-01-01', '2025-02-01', '2025-03-01', '2025-04-01', '2025-05-01', '2025-06-01', '2025-07-01', '2025-08-01', '2025-09-01', '2025-10-01', '2025-11-01', '2025-12-01']

CALENDAR VALUES
--------------------------------------------------
Holiday values:
holiday
NaN                 723
Republic Day          2
Independence Day      2
Diwali                2
Christmas             2
Name: count, dtype: int64

Promotion event values:
promotion_event
NaN                   656
Seasonal Promotion     75
Name: count, dtype: int64

CALENDAR EVENT COUNTS
--------------------------------------------------
Holiday populated: 8
Holiday missing: 72

,SKU,Product_Name,Category,Cost_Price,Selling_Price,Gross_Margin_Per_Unit
1,SKU002,Product 002,Home Decor,3867.09,3805.69,-61.40
6,SKU007,Product 007,Home Decor,7748.20,5114.09,-2634.11
8,SKU009,Product 009,Lighting,3121.06,2336.89,-784.17
9,SKU010,Product 010,Storage,3889.06,663.46,-3225.60
10,SKU011,Product 011,Furniture,1718.40,1444.56,-273.84
15,SKU016,Product 016,Furniture,3637.93,2166.82,-1471.11
17,SKU018,Product 018,Kitchen,5677.05,5575.41,-101.64
19,SKU020,Product 020,Storage,6700.01,3897.54,-2802.47
22,SKU023,Product 023,Kitchen,2484.54,1416.74,-1067.80
23,SKU024,Product 024,Lighting,5539.34,1768.87,-3770.47


In [8]:
# STEP 5 — Core EDA for FORESIGHT

# --------------------------------------------------
# 1. Create a weekly sales dataset
# --------------------------------------------------

weekly_sales = (
    sales
    .set_index("Date")
    .groupby("SKU")["Units_Sold"]
    .resample("W")
    .sum()
    .reset_index()
)

print("Weekly sales shape:", weekly_sales.shape)
display(weekly_sales.head())


# --------------------------------------------------
# 2. Overall sales summary
# --------------------------------------------------

print("\nOVERALL SALES")
print("-" * 50)

print("Total units sold:", sales["Units_Sold"].sum())
print("Total revenue: ₹", round(sales["Revenue"].sum(), 2))
print("Average units sold per day:", round(sales["Units_Sold"].mean(), 2))
print("Average daily revenue: ₹", round(sales["Revenue"].mean(), 2))


# --------------------------------------------------
# 3. Top 10 SKUs by total units sold
# --------------------------------------------------

top_skus = (
    sales.groupby("SKU")
    .agg(
        Total_Units=("Units_Sold", "sum"),
        Total_Revenue=("Revenue", "sum"),
        Avg_Daily_Units=("Units_Sold", "mean")
    )
    .sort_values("Total_Units", ascending=False)
)

print("\nTOP 10 SKUs BY UNITS SOLD")
print("-" * 50)

display(top_skus.head(10))


# --------------------------------------------------
# 4. Bottom 10 SKUs by total units sold
# --------------------------------------------------

print("\nBOTTOM 10 SKUs BY UNITS SOLD")
print("-" * 50)

display(top_skus.tail(10))


# --------------------------------------------------
# 5. SKU demand variability
# --------------------------------------------------

sku_variability = (
    sales.groupby("SKU")["Units_Sold"]
    .agg(
        Average_Daily_Demand="mean",
        Std_Daily_Demand="std",
        Minimum_Daily_Demand="min",
        Maximum_Daily_Demand="max"
    )
)

sku_variability["CV"] = (
    sku_variability["Std_Daily_Demand"] /
    sku_variability["Average_Daily_Demand"]
)

print("\nMOST VARIABLE SKUs")
print("-" * 50)

display(
    sku_variability
    .sort_values("CV", ascending=False)
    .head(10)
)


# --------------------------------------------------
# 6. Monthly demand trend
# --------------------------------------------------

monthly_sales = (
    sales
    .set_index("Date")
    .resample("MS")["Units_Sold"]
    .sum()
    .reset_index()
)

print("\nMONTHLY DEMAND")
print("-" * 50)

display(monthly_sales)


# --------------------------------------------------
# 7. Season performance
# --------------------------------------------------

sales_calendar = sales.merge(
    calendar[[
        "date",
        "season",
        "is_holiday",
        "promotion_event"
    ]],
    left_on="Date",
    right_on="date",
    how="left"
)

season_summary = (
    sales_calendar
    .groupby("season")
    .agg(
        Total_Units=("Units_Sold", "sum"),
        Average_Daily_Units=("Units_Sold", "mean"),
        Total_Revenue=("Revenue", "sum")
    )
    .sort_values("Total_Units", ascending=False)
)

print("\nSEASON PERFORMANCE")
print("-" * 50)

display(season_summary)


# --------------------------------------------------
# 8. Promotion vs non-promotion demand
# --------------------------------------------------

promotion_summary = (
    sales_calendar
    .groupby(sales_calendar["promotion_event"].notna())
    .agg(
        Total_Units=("Units_Sold", "sum"),
        Average_Units=("Units_Sold", "mean"),
        Total_Revenue=("Revenue", "sum")
    )
)

promotion_summary.index = [
    "No Promotion" if x is False else "Promotion"
    for x in promotion_summary.index
]

print("\nPROMOTION VS NON-PROMOTION")
print("-" * 50)

display(promotion_summary)


# --------------------------------------------------
# 9. Zero-demand days by SKU
# --------------------------------------------------

zero_demand = (
    sales.groupby("SKU")["Units_Sold"]
    .apply(lambda x: (x == 0).sum())
    .sort_values(ascending=False)
)

print("\nSKUs WITH MOST ZERO-DEMAND DAYS")
print("-" * 50)

display(zero_demand.head(10))

Weekly sales shape: (5250, 3)


,SKU,Date,Units_Sold
0,SKU001,2024-01-07,106
1,SKU001,2024-01-14,85
2,SKU001,2024-01-21,117
3,SKU001,2024-01-28,89
4,SKU001,2024-02-04,119



OVERALL SALES
--------------------------------------------------
Total units sold: 511810
Total revenue: ₹ 3096056707.22
Average units sold per day: 14.0
Average daily revenue: ₹ 84707.43

TOP 10 SKUs BY UNITS SOLD
--------------------------------------------------


,Total_Units,Total_Revenue,Avg_Daily_Units
SKU,,,
SKU012,19067,1.673962e+08,26.083447
SKU045,18612,1.428802e+08,25.461012
SKU018,18450,1.028663e+08,25.239398
SKU049,18373,9.596898e+07,25.134063
SKU037,18155,4.987941e+07,24.835841
SKU007,18129,9.271334e+07,24.800274
SKU027,17717,1.508886e+08,24.236662
SKU042,16616,1.528478e+08,22.730506
SKU026,16535,1.808853e+08,22.619699



BOTTOM 10 SKUs BY UNITS SOLD
--------------------------------------------------


...

,Total_Units,Average_Daily_Units,Total_Revenue
season,,,
Winter,122704,13.558453,7.442873e+08
Monsoon,118001,12.826196,7.140896e+08
Spring,101684,16.669508,6.141112e+08
Summer,97654,16.008852,5.900151e+08
Autumn,71767,11.765082,4.335536e+08



PROMOTION VS NON-PROMOTION
--------------------------------------------------


,Total_Units,Average_Units,Total_Revenue
No Promotion,442011,13.475945,2.674843e+09
Promotion,69799,18.613067,4.212141e+08



SKUs WITH MOST ZERO-DEMAND DAYS
--------------------------------------------------


SKU
SKU025    58
SKU011    55
SKU039    36
SKU015    15
SKU004    13
SKU036     8
SKU030     7
SKU003     4
SKU028     2
SKU038     1
Name: Units_Sold, dtype: int64

In [9]:
# STEP 6A — Correct Weekly Dataset + Seasonal-Naive Baseline

# --------------------------------------------------
# 1. Create weekly sales
# --------------------------------------------------

weekly_sales = (
    sales
    .set_index("Date")
    .groupby("SKU")["Units_Sold"]
    .resample("W-SUN")
    .sum()
    .reset_index()
)

weekly_sales = weekly_sales.sort_values(
    ["SKU", "Date"]
).reset_index(drop=True)


# --------------------------------------------------
# 2. Remove the incomplete final week
# --------------------------------------------------

last_complete_week = pd.Timestamp("2025-12-28")

weekly_sales = weekly_sales[
    weekly_sales["Date"] <= last_complete_week
].copy()


# --------------------------------------------------
# 3. Create 12-week test period
# --------------------------------------------------

all_weeks = sorted(weekly_sales["Date"].unique())

test_weeks = 12
test_start = all_weeks[-test_weeks]

train = weekly_sales[
    weekly_sales["Date"] < test_start
].copy()

test = weekly_sales[
    weekly_sales["Date"] >= test_start
].copy()


# --------------------------------------------------
# 4. Seasonal-naive prediction
# --------------------------------------------------

weekly_sales["Seasonal_Naive"] = (
    weekly_sales
    .groupby("SKU")["Units_Sold"]
    .shift(52)
)

test = weekly_sales[
    weekly_sales["Date"] >= test_start
].copy()

test = test.dropna(
    subset=["Seasonal_Naive"]
)


# --------------------------------------------------
# 5. Calculate WAPE
# --------------------------------------------------

wape = (
    (test["Units_Sold"] - test["Seasonal_Naive"])
    .abs()
    .sum()
    /
    test["Units_Sold"]
    .abs()
    .sum()
) * 100


# --------------------------------------------------
# 6. Display results
# --------------------------------------------------

print("CORRECTED WEEKLY DATASET")
print("-" * 50)

print("Weekly rows:", len(weekly_sales))
print("Unique SKUs:", weekly_sales["SKU"].nunique())
print("Complete weeks:", weekly_sales["Date"].nunique())

print("\nTRAIN / TEST SPLIT")
print("-" * 50)

print(
    "Training:",
    train["Date"].min().date(),
    "to",
    train["Date"].max().date()
)

print(
    "Testing:",
    test["Date"].min().date(),
    "to",
    test["Date"].max().date()
)

print("Test weeks:", test["Date"].nunique())


print("\nCORRECTED SEASONAL-NAIVE BASELINE")
print("-" * 50)

print("Test observations:", len(test))
print("Baseline WAPE:", round(wape, 2), "%")


print("\nSAMPLE PREDICTIONS")
print("-" * 50)

display(
    test[
        ["Date", "SKU", "Units_Sold", "Seasonal_Naive"]
    ].head(20)
)

CORRECTED WEEKLY DATASET
--------------------------------------------------
Weekly rows: 5200
Unique SKUs: 50
Complete weeks: 104

TRAIN / TEST SPLIT
--------------------------------------------------
Training: 2024-01-07 to 2025-10-05
Testing: 2025-10-12 to 2025-12-28
Test weeks: 12

CORRECTED SEASONAL-NAIVE BASELINE
--------------------------------------------------
Test observations: 600
Baseline WAPE: 11.51 %

SAMPLE PREDICTIONS
--------------------------------------------------


,Date,SKU,Units_Sold,Seasonal_Naive
92,2025-10-12,SKU001,82,82.0
93,2025-10-19,SKU001,82,88.0
94,2025-10-26,SKU001,78,89.0
95,2025-11-02,SKU001,93,83.0
96,2025-11-09,SKU001,92,99.0
97,2025-11-16,SKU001,87,80.0
98,2025-11-23,SKU001,84,88.0
99,2025-11-30,SKU001,84,68.0
100,2025-12-07,SKU001,88,83.0
101,2025-12-14,SKU001,89,94.0


In [10]:
# STEP 7 — Create Forecasting Features

# --------------------------------------------------
# 1. Start with the complete weekly dataset
# --------------------------------------------------

model_data = weekly_sales[
    ["SKU", "Date", "Units_Sold"]
].copy()

model_data = model_data.sort_values(
    ["SKU", "Date"]
).reset_index(drop=True)


# --------------------------------------------------
# 2. Demand lag features
# --------------------------------------------------

model_data["lag_1"] = (
    model_data.groupby("SKU")["Units_Sold"].shift(1)
)

model_data["lag_2"] = (
    model_data.groupby("SKU")["Units_Sold"].shift(2)
)

model_data["lag_4"] = (
    model_data.groupby("SKU")["Units_Sold"].shift(4)
)

model_data["lag_8"] = (
    model_data.groupby("SKU")["Units_Sold"].shift(8)
)

model_data["lag_12"] = (
    model_data.groupby("SKU")["Units_Sold"].shift(12)
)

model_data["lag_52"] = (
    model_data.groupby("SKU")["Units_Sold"].shift(52)
)


# --------------------------------------------------
# 3. Rolling demand features
# --------------------------------------------------
# Shift first so the current week's demand
# is NOT included in its own features.

model_data["rolling_mean_4"] = (
    model_data.groupby("SKU")["Units_Sold"]
    .transform(lambda x: x.shift(1).rolling(4).mean())
)

model_data["rolling_mean_8"] = (
    model_data.groupby("SKU")["Units_Sold"]
    .transform(lambda x: x.shift(1).rolling(8).mean())
)

model_data["rolling_mean_12"] = (
    model_data.groupby("SKU")["Units_Sold"]
    .transform(lambda x: x.shift(1).rolling(12).mean())
)


# --------------------------------------------------
# 4. Calendar features
# --------------------------------------------------

calendar_features = calendar[
    [
        "date",
        "month",
        "quarter",
        "week",
        "season",
        "is_holiday",
        "promotion_event"
    ]
].copy()

# Convert weekly date to the corresponding
# week's calendar information.
calendar_features["Date"] = (
    pd.to_datetime(calendar_features["date"])
    .dt.to_period("W-SUN")
    .dt.end_time
    .dt.normalize()
)

calendar_features = calendar_features.drop(
    columns=["date"]
).drop_duplicates("Date")


# --------------------------------------------------
# 5. Merge calendar information
# --------------------------------------------------

model_data = model_data.merge(
    calendar_features,
    on="Date",
    how="left"
)


# --------------------------------------------------
# 6. Convert categorical features to numeric
# --------------------------------------------------

model_data = pd.get_dummies(
    model_data,
    columns=["SKU", "season", "promotion_event"],
    drop_first=False
)


# --------------------------------------------------
# 7. Check the result
# --------------------------------------------------

print("FORECASTING DATASET")
print("-" * 50)

print("Shape:", model_data.shape)
print("Missing values:", model_data.isna().sum().sum())

print("\nFEATURES:")
print(model_data.columns.tolist())

display(model_data.head())

FORECASTING DATASET
--------------------------------------------------
Shape: (5200, 70)
Missing values: 5150

FEATURES:
['Date', 'Units_Sold', 'lag_1', 'lag_2', 'lag_4', 'lag_8', 'lag_12', 'lag_52', 'rolling_mean_4', 'rolling_mean_8', 'rolling_mean_12', 'month', 'quarter', 'week', 'is_holiday', 'SKU_SKU001', 'SKU_SKU002', 'SKU_SKU003', 'SKU_SKU004', 'SKU_SKU005', 'SKU_SKU006', 'SKU_SKU007', 'SKU_SKU008', 'SKU_SKU009', 'SKU_SKU010', 'SKU_SKU011', 'SKU_SKU012', 'SKU_SKU013', 'SKU_SKU014', 'SKU_SKU015', 'SKU_SKU016', 'SKU_SKU017', 'SKU_SKU018', 'SKU_SKU019', 'SKU_SKU020', 'SKU_SKU021', 'SKU_SKU022', 'SKU_SKU023', 'SKU_SKU024', 'SKU_SKU025', 'SKU_SKU026', 'SKU_SKU027', 'SKU_SKU028', 'SKU_SKU029', 'SKU_SKU030', 'SKU_SKU031', 'SKU_SKU032', 'SKU_SKU033', 'SKU_SKU034', 'SKU_SKU035', 'SKU_SKU036', 'SKU_SKU037', 'SKU_SKU038', 'SKU_SKU039', 'SKU_SKU040', 'SKU_SKU041', 'SKU_SKU042', 'SKU_SKU043', 'SKU_SKU044', 'SKU_SKU045', 'SKU_SKU046', 'SKU_SKU047', 'SKU_SKU048', 'SKU_SKU049', 'SKU_SKU050', 'se

,Date,Units_Sold,lag_1,lag_2,lag_4,lag_8,lag_12,lag_52,rolling_mean_4,rolling_mean_8,...,SKU_SKU046,SKU_SKU047,SKU_SKU048,SKU_SKU049,SKU_SKU050,season_Autumn,season_Monsoon,season_Spring,season_Summer,season_Winter
0,2024-01-07,106,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,False,False,False,False,False,False,False,False,False,True
1,2024-01-14,85,106.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,False,False,False,False,False,False,False,False,False,True
2,2024-01-21,117,85.0,106.0,NaN,NaN,NaN,NaN,NaN,NaN,...,False,False,False,False,False,False,False,False,False,True
3,2024-01-28,89,117.0,85.0,NaN,NaN,NaN,NaN,NaN,NaN,...,False,False,False,False,False,False,False,False,False,True
4,2024-02-04,119,89.0,117.0,106.0,NaN,NaN,NaN,99.25,NaN,...,False,False,False,False,False,False,False,False,False,True


In [11]:
# STEP 7A — Fix Promotion Feature
# --------------------------------

# Rebuild calendar features with an explicit
# "No Promotion" category for missing values.

calendar_features = calendar[
    [
        "date",
        "month",
        "quarter",
        "week",
        "season",
        "is_holiday",
        "promotion_event"
    ]
].copy()

calendar_features["promotion_event"] = (
    calendar_features["promotion_event"]
    .fillna("No Promotion")
)

calendar_features["Date"] = (
    pd.to_datetime(calendar_features["date"])
    .dt.to_period("W-SUN")
    .dt.end_time
    .dt.normalize()
)

calendar_features = (
    calendar_features
    .drop(columns=["date"])
    .drop_duplicates("Date")
)

# Remove the old calendar columns from model_data
# before merging the corrected versions.
model_data = model_data.drop(
    columns=[
        "month",
        "quarter",
        "week",
        "is_holiday",
        "season_Autumn",
        "season_Monsoon",
        "season_Spring",
        "season_Summer",
        "season_Winter"
    ],
    errors="ignore"
)

# Add corrected calendar information
model_data = model_data.merge(
    calendar_features,
    on="Date",
    how="left"
)

# One-hot encode the corrected categorical columns
model_data = pd.get_dummies(
    model_data,
    columns=["season", "promotion_event"],
    drop_first=False
)

print("UPDATED FORECASTING DATASET")
print("-" * 50)

print("Shape:", model_data.shape)
print("Missing values:", model_data.isna().sum().sum())

print("\nPromotion features:")
print(
    [col for col in model_data.columns
     if "promotion_event" in col]
)

UPDATED FORECASTING DATASET
--------------------------------------------------
Shape: (5200, 71)
Missing values: 5150

Promotion features:
['promotion_event_No Promotion']


In [12]:
# STEP 7B — Check Weekly Promotion Information
# ----------------------------------------------

calendar_check = calendar.copy()

calendar_check["Date"] = (
    pd.to_datetime(calendar_check["date"])
)

calendar_check["Week_End"] = (
    calendar_check["Date"]
    .dt.to_period("W-SUN")
    .dt.end_time
    .dt.normalize()
)

# Treat missing promotion values as "No Promotion"
calendar_check["promotion_event"] = (
    calendar_check["promotion_event"]
    .fillna("No Promotion")
)

# Count promotion days in each week
weekly_promotion_check = (
    calendar_check
    .groupby("Week_End")["promotion_event"]
    .apply(lambda x: (x != "No Promotion").sum())
    .reset_index(name="promotion_days")
)

print("WEEKS WITH PROMOTIONS")
print("-" * 50)

print(
    weekly_promotion_check[
        weekly_promotion_check["promotion_days"] > 0
    ].head(20)
)

print(
    "\nTotal weeks with at least one promotion:",
    (
        weekly_promotion_check["promotion_days"] > 0
    ).sum()
)

WEEKS WITH PROMOTIONS
--------------------------------------------------
     Week_End  promotion_days
8  2024-03-03               2
9  2024-03-10               2
10 2024-03-17               2
11 2024-03-24               2
12 2024-03-31               2
21 2024-06-02               2
22 2024-06-09               2
23 2024-06-16               2
24 2024-06-23               2
25 2024-06-30               2
34 2024-09-01               1
35 2024-09-08               2
36 2024-09-15               2
37 2024-09-22               2
38 2024-09-29               2
43 2024-11-03               2
44 2024-11-10               2
45 2024-11-17               2
46 2024-11-24               2
47 2024-12-01               1

Total weeks with at least one promotion: 39


In [13]:
# STEP 7C — Build Correct Weekly Calendar Features
# ------------------------------------------------

calendar_weekly = calendar.copy()

calendar_weekly["date"] = pd.to_datetime(
    calendar_weekly["date"]
)

# Assign every day to its Sunday-ending week
calendar_weekly["Date"] = (
    calendar_weekly["date"]
    .dt.to_period("W-SUN")
    .dt.end_time
    .dt.normalize()
)

# Create one row per week
weekly_calendar = (
    calendar_weekly
    .groupby("Date")
    .agg(
        month=("month", "first"),
        quarter=("quarter", "first"),
        week=("week", "first"),
        season=("season", "first"),
        is_holiday=("is_holiday", "max"),
        promotion_days=("promotion_event", lambda x:
                        x.notna().sum())
    )
    .reset_index()
)

# If at least one day had a promotion,
# classify the entire week as a promotion week.
weekly_calendar["promotion_event"] = np.where(
    weekly_calendar["promotion_days"] > 0,
    "Seasonal Promotion",
    "No Promotion"
)

# We no longer need the count itself as a model feature
weekly_calendar = weekly_calendar.drop(
    columns=["promotion_days"]
)

# Remove the old calendar features
model_data = model_data.drop(
    columns=[
        "month",
        "quarter",
        "week",
        "is_holiday",
        "season_Autumn",
        "season_Monsoon",
        "season_Spring",
        "season_Summer",
        "season_Winter",
        "promotion_event_No Promotion"
    ],
    errors="ignore"
)

# Merge the corrected weekly calendar
model_data = model_data.merge(
    weekly_calendar,
    on="Date",
    how="left"
)

# One-hot encode categorical variables
model_data = pd.get_dummies(
    model_data,
    columns=["season", "promotion_event"],
    drop_first=False
)

print("FINAL FORECASTING DATASET")
print("-" * 50)

print("Shape:", model_data.shape)

print(
    "Missing values:",
    model_data.isna().sum().sum()
)

print("\nPromotion features:")
print(
    [
        col for col in model_data.columns
        if "promotion_event" in col
    ]
)

print("\nPromotion weeks:")
print(
    weekly_calendar[
        weekly_calendar["promotion_event"]
        == "Seasonal Promotion"
    ].shape[0]
)

display(model_data.head())

FINAL FORECASTING DATASET
--------------------------------------------------
Shape: (5200, 72)
Missing values: 5150

Promotion features:
['promotion_event_No Promotion', 'promotion_event_Seasonal Promotion']

Promotion weeks:
39


,Date,Units_Sold,lag_1,lag_2,lag_4,lag_8,lag_12,lag_52,rolling_mean_4,rolling_mean_8,...,quarter,week,is_holiday,season_Autumn,season_Monsoon,season_Spring,season_Summer,season_Winter,promotion_event_No Promotion,promotion_event_Seasonal Promotion
0,2024-01-07,106,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Q1,1,0,False,False,False,False,True,True,False
1,2024-01-14,85,106.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Q1,2,0,False,False,False,False,True,True,False
2,2024-01-21,117,85.0,106.0,NaN,NaN,NaN,NaN,NaN,NaN,...,Q1,3,0,False,False,False,False,True,True,False
3,2024-01-28,89,117.0,85.0,NaN,NaN,NaN,NaN,NaN,NaN,...,Q1,4,1,False,False,False,False,True,True,False
4,2024-02-04,119,89.0,117.0,106.0,NaN,NaN,NaN,99.25,NaN,...,Q1,5,0,False,False,False,False,True,True,False


In [14]:
# STEP 8 — Prepare Train/Test Data
# --------------------------------

# Official test period from our baseline
test_start = pd.Timestamp("2025-10-12")
test_end = pd.Timestamp("2025-12-28")

# Keep only rows where all required forecasting
# features are available.
required_features = [
    "lag_1",
    "lag_2",
    "lag_4",
    "lag_8",
    "lag_12",
    "lag_52",
    "rolling_mean_4",
    "rolling_mean_8",
    "rolling_mean_12"
]

model_ready = model_data.dropna(
    subset=required_features
).copy()

# Split using the exact same test period
train_data = model_ready[
    model_ready["Date"] < test_start
].copy()

test_data = model_ready[
    (model_ready["Date"] >= test_start) &
    (model_ready["Date"] <= test_end)
].copy()

print("MODEL-READY DATA")
print("-" * 50)

print("Total rows:", len(model_ready))

print("\nTRAINING DATA")
print("Rows:", len(train_data))
print(
    "Date range:",
    train_data["Date"].min(),
    "to",
    train_data["Date"].max()
)

print("\nTEST DATA")
print("Rows:", len(test_data))
print(
    "Date range:",
    test_data["Date"].min(),
    "to",
    test_data["Date"].max()
)

print("\nMissing values in train:",
      train_data.isna().sum().sum())

print("Missing values in test:",
      test_data.isna().sum().sum())

MODEL-READY DATA
--------------------------------------------------
Total rows: 2600

TRAINING DATA
Rows: 2000
Date range: 2025-01-05 00:00:00 to 2025-10-05 00:00:00

TEST DATA
Rows: 600
Date range: 2025-10-12 00:00:00 to 2025-12-28 00:00:00

Missing values in train: 0
Missing values in test: 0


In [15]:
# STEP 8B — Separate Features and Target
# ---------------------------------------

# Columns that should NOT be used as model features
# because Date is only used for identifying the time period.
drop_columns = [
    "Units_Sold",
    "Date"
]

# Create feature matrices
X_train = train_data.drop(
    columns=drop_columns
)

X_test = test_data.drop(
    columns=drop_columns
)

# Create target vectors
y_train = train_data["Units_Sold"]
y_test = test_data["Units_Sold"]

print("TRAINING FEATURES")
print("-" * 50)
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("\nTEST FEATURES")
print("-" * 50)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

print("\nFeature columns:")
print(X_train.columns.tolist())

TRAINING FEATURES
--------------------------------------------------
X_train shape: (2000, 70)
y_train shape: (2000,)

TEST FEATURES
--------------------------------------------------
X_test shape: (600, 70)
y_test shape: (600,)

Feature columns:
['lag_1', 'lag_2', 'lag_4', 'lag_8', 'lag_12', 'lag_52', 'rolling_mean_4', 'rolling_mean_8', 'rolling_mean_12', 'SKU_SKU001', 'SKU_SKU002', 'SKU_SKU003', 'SKU_SKU004', 'SKU_SKU005', 'SKU_SKU006', 'SKU_SKU007', 'SKU_SKU008', 'SKU_SKU009', 'SKU_SKU010', 'SKU_SKU011', 'SKU_SKU012', 'SKU_SKU013', 'SKU_SKU014', 'SKU_SKU015', 'SKU_SKU016', 'SKU_SKU017', 'SKU_SKU018', 'SKU_SKU019', 'SKU_SKU020', 'SKU_SKU021', 'SKU_SKU022', 'SKU_SKU023', 'SKU_SKU024', 'SKU_SKU025', 'SKU_SKU026', 'SKU_SKU027', 'SKU_SKU028', 'SKU_SKU029', 'SKU_SKU030', 'SKU_SKU031', 'SKU_SKU032', 'SKU_SKU033', 'SKU_SKU034', 'SKU_SKU035', 'SKU_SKU036', 'SKU_SKU037', 'SKU_SKU038', 'SKU_SKU039', 'SKU_SKU040', 'SKU_SKU041', 'SKU_SKU042', 'SKU_SKU043', 'SKU_SKU044', 'SKU_SKU045', 'SKU_SKU046

In [16]:
# Convert quarter from strings (Q1, Q2, Q3, Q4) to numbers (1, 2, 3, 4)

for df in [X_train, X_test]:
    df["quarter"] = (
        df["quarter"]
        .str.replace("Q", "", regex=False)
        .astype(int)
    )

print(X_train["quarter"].unique())
print(X_test["quarter"].unique())

[4 1 2 3]
[4]


In [17]:
# STEP 9 — Train First Forecasting Model
# --------------------------------------

from sklearn.ensemble import RandomForestRegressor

# Create the model
rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

# Train ONLY on the training data
rf_model.fit(
    X_train,
    y_train
)

# Predict the held-out test period
y_pred = rf_model.predict(X_test)

# Make sure predictions cannot be negative
y_pred = np.maximum(y_pred, 0)

# Calculate WAPE
wape = (
    np.abs(y_test - y_pred).sum()
    / y_test.sum()
) * 100

print("RANDOM FOREST FORECAST RESULTS")
print("-" * 50)

print("Test observations:", len(y_test))
print(f"WAPE: {wape:.2f}%")

print("\nBaseline comparison")
print("-" * 50)
print("Seasonal-naive WAPE: 11.51%")
print(f"Random Forest WAPE:  {wape:.2f}%")

if wape < 11.51:
    print("\n✅ MODEL BEATS THE SEASONAL-NAIVE BASELINE!")
else:
    print("\n❌ MODEL DOES NOT BEAT THE BASELINE YET.")

RANDOM FOREST FORECAST RESULTS
--------------------------------------------------
Test observations: 600
WAPE: 10.11%

Baseline comparison
--------------------------------------------------
Seasonal-naive WAPE: 11.51%
Random Forest WAPE:  10.11%

✅ MODEL BEATS THE SEASONAL-NAIVE BASELINE!


In [18]:
# Save Random Forest predictions alongside the actual demand

# Reconstruct SKU from the one-hot encoded SKU columns
sku_columns = [
    col for col in X_test.columns
    if col.startswith("SKU_")
]

test_results = test_data[
    ["Date", "Units_Sold"]
].copy()

test_results["SKU"] = (
    X_test[sku_columns]
    .idxmax(axis=1)
    .str.replace("SKU_", "", regex=False)
)

test_results["Forecast_Units"] = y_pred

# Put columns in a clean order
test_results = test_results[
    ["SKU", "Date", "Units_Sold", "Forecast_Units"]
]

print(test_results.head(10))
print()
print("Test result shape:", test_results.shape)

        SKU       Date  Units_Sold  Forecast_Units
92   SKU001 2025-10-12          82       83.838261
93   SKU001 2025-10-19          82       82.005937
94   SKU001 2025-10-26          78       80.153353
95   SKU001 2025-11-02          93       76.286473
96   SKU001 2025-11-09          92       83.638671
97   SKU001 2025-11-16          87       81.183546
98   SKU001 2025-11-23          84       81.503385
99   SKU001 2025-11-30          84       80.844128
100  SKU001 2025-12-07          88       79.321284
101  SKU001 2025-12-14          89       82.784611

Test result shape: (600, 4)


In [19]:
print("Unique SKUs:", test_results["SKU"].nunique())
print("Rows:", len(test_results))
print(
    "Negative forecasts:",
    (test_results["Forecast_Units"] < 0).sum()
)

Unique SKUs: 50
Rows: 600
Negative forecasts: 0


In [20]:
# Get the latest inventory snapshot for the 50 project SKUs

latest_inventory = (
    project_inventory
    .sort_values(["SKU", "Snapshot_Date"])
    .groupby("SKU")
    .tail(1)
    .copy()
)

# Keep only the fields needed for risk scoring
latest_inventory = latest_inventory[
    [
        "SKU",
        "Snapshot_Date",
        "Current_Stock",
        "On_Order",
        "Lead_Time_Days",
        "Safety_Stock",
        "Reorder_Point",
        "Inventory_Value"
    ]
]

print(latest_inventory.head())
print()
print("Rows:", len(latest_inventory))
print("Unique SKUs:", latest_inventory["SKU"].nunique())
print("Latest snapshot:", latest_inventory["Snapshot_Date"].min())
print("Latest snapshot:", latest_inventory["Snapshot_Date"].max())

         SKU Snapshot_Date  Current_Stock  On_Order  Lead_Time_Days  \
4600  SKU001    2025-12-01            771       147               4   
4601  SKU002    2025-12-01            526        29               3   
4602  SKU003    2025-12-01            349        12               3   
4603  SKU004    2025-12-01            259       254               6   
4604  SKU005    2025-12-01            471       291               6   

      Safety_Stock  Reorder_Point  Inventory_Value  
4600           159            289       2815160.01  
4601            94            149       2974319.60  
4602            88            123       2145941.67  
4603            98            196        321905.92  
4604           129            245       3500674.53  

Rows: 50
Unique SKUs: 50
Latest snapshot: 2025-12-01 00:00:00
Latest snapshot: 2025-12-01 00:00:00


In [21]:
# Combine forecast results with the latest inventory position

risk_data = test_results.merge(
    latest_inventory,
    on="SKU",
    how="left"
)

print(risk_data.head(10))
print()
print("Shape:", risk_data.shape)
print("Missing inventory values:", risk_data.isna().sum().sum())

      SKU       Date  Units_Sold  Forecast_Units Snapshot_Date  Current_Stock  \
0  SKU001 2025-10-12          82       83.838261    2025-12-01            771   
1  SKU001 2025-10-19          82       82.005937    2025-12-01            771   
2  SKU001 2025-10-26          78       80.153353    2025-12-01            771   
3  SKU001 2025-11-02          93       76.286473    2025-12-01            771   
4  SKU001 2025-11-09          92       83.638671    2025-12-01            771   
5  SKU001 2025-11-16          87       81.183546    2025-12-01            771   
6  SKU001 2025-11-23          84       81.503385    2025-12-01            771   
7  SKU001 2025-11-30          84       80.844128    2025-12-01            771   
8  SKU001 2025-12-07          88       79.321284    2025-12-01            771   
9  SKU001 2025-12-14          89       82.784611    2025-12-01            771   

   On_Order  Lead_Time_Days  Safety_Stock  Reorder_Point  Inventory_Value  
0       147               4     

In [22]:
# Use only forecast weeks after the latest inventory snapshot

planning_start = pd.Timestamp("2025-12-07")

planning_data = risk_data[
    risk_data["Date"] >= planning_start
].copy()

print("Planning horizon:")
print(planning_data["Date"].min(), "to", planning_data["Date"].max())

print()
print("Rows:", len(planning_data))
print("Unique SKUs:", planning_data["SKU"].nunique())

Planning horizon:
2025-12-07 00:00:00 to 2025-12-28 00:00:00

Rows: 200
Unique SKUs: 50


In [23]:
# Estimate demand during each SKU's replenishment lead time

planning_data["Lead_Time_Weeks"] = (
    planning_data["Lead_Time_Days"] / 7
)

planning_data["Lead_Time_Demand"] = (
    planning_data["Forecast_Units"] *
    planning_data["Lead_Time_Weeks"]
)

print(
    planning_data[
        [
            "SKU",
            "Date",
            "Forecast_Units",
            "Lead_Time_Days",
            "Lead_Time_Weeks",
            "Lead_Time_Demand",
            "Current_Stock",
            "On_Order",
            "Safety_Stock",
            "Reorder_Point"
        ]
    ].head(10)
)

       SKU       Date  Forecast_Units  Lead_Time_Days  Lead_Time_Weeks  \
8   SKU001 2025-12-07       79.321284               4         0.571429   
9   SKU001 2025-12-14       82.784611               4         0.571429   
10  SKU001 2025-12-21       81.107468               4         0.571429   
11  SKU001 2025-12-28       85.246719               4         0.571429   
20  SKU002 2025-12-07       60.678925               3         0.428571   
21  SKU002 2025-12-14       57.132529               3         0.428571   
22  SKU002 2025-12-21       57.395606               3         0.428571   
23  SKU002 2025-12-28       60.473185               3         0.428571   
32  SKU003 2025-12-07       34.793999               3         0.428571   
33  SKU003 2025-12-14       34.008989               3         0.428571   

    Lead_Time_Demand  Current_Stock  On_Order  Safety_Stock  Reorder_Point  
8          45.326448            771       147           159            289  
9          47.305492           

In [24]:
# Calculate cumulative forecast demand for each SKU

planning_data = planning_data.sort_values(
    ["SKU", "Date"]
).copy()

planning_data["Cumulative_Forecast"] = (
    planning_data
    .groupby("SKU")["Forecast_Units"]
    .cumsum()
)

print(
    planning_data[
        [
            "SKU",
            "Date",
            "Forecast_Units",
            "Cumulative_Forecast",
            "Current_Stock",
            "On_Order"
        ]
    ].head(12)
)

       SKU       Date  Forecast_Units  Cumulative_Forecast  Current_Stock  \
8   SKU001 2025-12-07       79.321284            79.321284            771   
9   SKU001 2025-12-14       82.784611           162.105895            771   
10  SKU001 2025-12-21       81.107468           243.213364            771   
11  SKU001 2025-12-28       85.246719           328.460083            771   
20  SKU002 2025-12-07       60.678925            60.678925            526   
21  SKU002 2025-12-14       57.132529           117.811455            526   
22  SKU002 2025-12-21       57.395606           175.207061            526   
23  SKU002 2025-12-28       60.473185           235.680245            526   
32  SKU003 2025-12-07       34.793999            34.793999            349   
33  SKU003 2025-12-14       34.008989            68.802988            349   
34  SKU003 2025-12-21       33.611119           102.414107            349   
35  SKU003 2025-12-28       31.361034           133.775140            349   

In [25]:
# Calculate total available supply

planning_data["Available_Supply"] = (
    planning_data["Current_Stock"] +
    planning_data["On_Order"]
)

# Calculate projected inventory balance
planning_data["Projected_Balance"] = (
    planning_data["Available_Supply"] -
    planning_data["Cumulative_Forecast"]
)

print(
    planning_data[
        [
            "SKU",
            "Date",
            "Cumulative_Forecast",
            "Available_Supply",
            "Projected_Balance"
        ]
    ].head(12)
)

       SKU       Date  Cumulative_Forecast  Available_Supply  \
8   SKU001 2025-12-07            79.321284               918   
9   SKU001 2025-12-14           162.105895               918   
10  SKU001 2025-12-21           243.213364               918   
11  SKU001 2025-12-28           328.460083               918   
20  SKU002 2025-12-07            60.678925               555   
21  SKU002 2025-12-14           117.811455               555   
22  SKU002 2025-12-21           175.207061               555   
23  SKU002 2025-12-28           235.680245               555   
32  SKU003 2025-12-07            34.793999               361   
33  SKU003 2025-12-14            68.802988               361   
34  SKU003 2025-12-21           102.414107               361   
35  SKU003 2025-12-28           133.775140               361   

    Projected_Balance  
8          838.678716  
9          755.894105  
10         674.786636  
11         589.539917  
20         494.321075  
21         437.188545  

In [26]:
# Look at each SKU's projected inventory position
# at the end of the 4-week planning horizon.

final_week = (
    planning_data[
        planning_data["Date"] == planning_data["Date"].max()
    ]
    .copy()
    .sort_values("Projected_Balance")
)

print(
    final_week[
        [
            "SKU",
            "Forecast_Units",
            "Cumulative_Forecast",
            "Available_Supply",
            "Projected_Balance",
            "Safety_Stock",
            "Reorder_Point"
        ]
    ].to_string(index=False)
)

   SKU  Forecast_Units  Cumulative_Forecast  Available_Supply  Projected_Balance  Safety_Stock  Reorder_Point
SKU012      158.283041           626.982200               107        -519.982200            16             58
SKU034      122.246814           493.428788               137        -356.428788            34             58
SKU042      154.061713           556.732443               210        -346.732443            41             88
SKU026      137.297385           543.885015               230        -313.885015            31             59
SKU018      147.190375           620.434982               337        -283.434982            66            184
SKU031      101.827885           406.864183               133        -273.864183            28            113
SKU040       90.417663           349.721358                96        -253.721358            36             93
SKU010       97.190185           388.138295               153        -235.138295            22             60
SKU017    

In [27]:
# Create transparent inventory risk signals

planning_data["Stockout_Risk"] = (
    planning_data["Projected_Balance"] < 0
)

planning_data["Below_Safety_Stock"] = (
    planning_data["Projected_Balance"] < planning_data["Safety_Stock"]
)

planning_data["Below_Reorder_Point"] = (
    planning_data["Projected_Balance"] < planning_data["Reorder_Point"]
)

# Excess inventory above the expected 4-week demand
planning_data["Excess_Inventory"] = (
    planning_data["Projected_Balance"] >
    planning_data["Cumulative_Forecast"] +
    planning_data["Safety_Stock"]
)

print(
    planning_data[
        [
            "SKU",
            "Date",
            "Projected_Balance",
            "Safety_Stock",
            "Reorder_Point",
            "Stockout_Risk",
            "Below_Safety_Stock",
            "Below_Reorder_Point",
            "Excess_Inventory"
        ]
    ].head(20)
)

       SKU       Date  Projected_Balance  Safety_Stock  Reorder_Point  \
8   SKU001 2025-12-07         838.678716           159            289   
9   SKU001 2025-12-14         755.894105           159            289   
10  SKU001 2025-12-21         674.786636           159            289   
11  SKU001 2025-12-28         589.539917           159            289   
20  SKU002 2025-12-07         494.321075            94            149   
21  SKU002 2025-12-14         437.188545            94            149   
22  SKU002 2025-12-21         379.792939            94            149   
23  SKU002 2025-12-28         319.319755            94            149   
32  SKU003 2025-12-07         326.206001            88            123   
33  SKU003 2025-12-14         292.197012            88            123   
34  SKU003 2025-12-21         258.585893            88            123   
35  SKU003 2025-12-28         227.224860            88            123   
44  SKU004 2025-12-07         485.489950           

In [28]:
# Summarize the number of SKU-week observations
# triggering each inventory risk signal.

risk_signal_summary = pd.Series({
    "Stockout Risk": planning_data["Stockout_Risk"].sum(),
    "Below Safety Stock": planning_data["Below_Safety_Stock"].sum(),
    "Below Reorder Point": planning_data["Below_Reorder_Point"].sum(),
    "Excess Inventory": planning_data["Excess_Inventory"].sum()
})

print(risk_signal_summary)
print()
print("Total SKU-week observations:", len(planning_data))

Stockout Risk          38
Below Safety Stock     53
Below Reorder Point    82
Excess Inventory       93
dtype: int64

Total SKU-week observations: 200


In [29]:
# Create a final business-friendly risk classification.
# Priority: Stockout > Replenishment > Overstock > Healthy

def classify_risk(row):
    if row["Stockout_Risk"]:
        return "Stockout Risk"
    elif row["Below_Reorder_Point"]:
        return "Replenishment Risk"
    elif row["Excess_Inventory"]:
        return "Overstock Risk"
    else:
        return "Healthy"


planning_data["Risk_Level"] = planning_data.apply(
    classify_risk,
    axis=1
)

print(
    planning_data["Risk_Level"]
    .value_counts()
)

Risk_Level
Overstock Risk        91
Replenishment Risk    44
Stockout Risk         38
Healthy               27
Name: count, dtype: int64


In [30]:
# Convert risk levels into recommended operational actions.

action_map = {
    "Stockout Risk": "Prioritize replenishment",
    "Replenishment Risk": "Review reorder / place order",
    "Overstock Risk": "Consider markdown / reduce replenishment",
    "Healthy": "No immediate action"
}

planning_data["Recommended_Action"] = (
    planning_data["Risk_Level"]
    .map(action_map)
)

print(
    planning_data[
        [
            "SKU",
            "Date",
            "Risk_Level",
            "Projected_Balance",
            "Recommended_Action"
        ]
    ].head(20).to_string(index=False)
)

   SKU       Date     Risk_Level  Projected_Balance                       Recommended_Action
SKU001 2025-12-07 Overstock Risk         838.678716 Consider markdown / reduce replenishment
SKU001 2025-12-14 Overstock Risk         755.894105 Consider markdown / reduce replenishment
SKU001 2025-12-21 Overstock Risk         674.786636 Consider markdown / reduce replenishment
SKU001 2025-12-28 Overstock Risk         589.539917 Consider markdown / reduce replenishment
SKU002 2025-12-07 Overstock Risk         494.321075 Consider markdown / reduce replenishment
SKU002 2025-12-14 Overstock Risk         437.188545 Consider markdown / reduce replenishment
SKU002 2025-12-21 Overstock Risk         379.792939 Consider markdown / reduce replenishment
SKU002 2025-12-28        Healthy         319.319755                      No immediate action
SKU003 2025-12-07 Overstock Risk         326.206001 Consider markdown / reduce replenishment
SKU003 2025-12-14 Overstock Risk         292.197012 Consider markdown 

In [31]:
# Calculate the quantity associated with each risk.

planning_data["Shortage_Units"] = (
    -planning_data["Projected_Balance"]
).clip(lower=0)

planning_data["Excess_Units"] = (
    planning_data["Projected_Balance"]
    - planning_data["Cumulative_Forecast"]
    - planning_data["Safety_Stock"]
).clip(lower=0)

print(
    planning_data[
        [
            "SKU",
            "Date",
            "Risk_Level",
            "Forecast_Units",
            "Cumulative_Forecast",
            "Available_Supply",
            "Projected_Balance",
            "Shortage_Units",
            "Excess_Units",
            "Recommended_Action"
        ]
    ].head(20).to_string(index=False)
)

   SKU       Date     Risk_Level  Forecast_Units  Cumulative_Forecast  Available_Supply  Projected_Balance  Shortage_Units  Excess_Units                       Recommended_Action
SKU001 2025-12-07 Overstock Risk       79.321284            79.321284               918         838.678716             0.0    600.357431 Consider markdown / reduce replenishment
SKU001 2025-12-14 Overstock Risk       82.784611           162.105895               918         755.894105             0.0    434.788210 Consider markdown / reduce replenishment
SKU001 2025-12-21 Overstock Risk       81.107468           243.213364               918         674.786636             0.0    272.573273 Consider markdown / reduce replenishment
SKU001 2025-12-28 Overstock Risk       85.246719           328.460083               918         589.539917             0.0    102.079834 Consider markdown / reduce replenishment
SKU002 2025-12-07 Overstock Risk       60.678925            60.678925               555         494.321075    

In [32]:
# Keep one final inventory position per SKU:
# the end of the 4-week planning horizon.

final_risk = (
    planning_data
    .sort_values(["SKU", "Date"])
    .groupby("SKU")
    .tail(1)
    .copy()
)

final_risk = final_risk.sort_values(
    "Projected_Balance"
)

print(
    final_risk[
        [
            "SKU",
            "Risk_Level",
            "Cumulative_Forecast",
            "Available_Supply",
            "Projected_Balance",
            "Shortage_Units",
            "Excess_Units",
            "Safety_Stock",
            "Reorder_Point",
            "Recommended_Action"
        ]
    ].to_string(index=False)
)

   SKU         Risk_Level  Cumulative_Forecast  Available_Supply  Projected_Balance  Shortage_Units  Excess_Units  Safety_Stock  Reorder_Point                       Recommended_Action
SKU012      Stockout Risk           626.982200               107        -519.982200      519.982200      0.000000            16             58                 Prioritize replenishment
SKU034      Stockout Risk           493.428788               137        -356.428788      356.428788      0.000000            34             58                 Prioritize replenishment
SKU042      Stockout Risk           556.732443               210        -346.732443      346.732443      0.000000            41             88                 Prioritize replenishment
SKU026      Stockout Risk           543.885015               230        -313.885015      313.885015      0.000000            31             59                 Prioritize replenishment
SKU018      Stockout Risk           620.434982               337        -283.434

In [33]:
# Bring product pricing information into the final risk table.

final_risk = final_risk.merge(
    sku[
        [
            "SKU",
            "Product_Name",
            "Category",
            "Subcategory",
            "Cost_Price",
            "Selling_Price"
        ]
    ],
    on="SKU",
    how="left"
)

print(
    final_risk[
        [
            "SKU",
            "Product_Name",
            "Risk_Level",
            "Shortage_Units",
            "Excess_Units",
            "Cost_Price",
            "Selling_Price"
        ]
    ].head(20).to_string(index=False)
)

print()
print("Missing Cost Price:", final_risk["Cost_Price"].isna().sum())
print("Missing Selling Price:", final_risk["Selling_Price"].isna().sum())

   SKU Product_Name         Risk_Level  Shortage_Units  Excess_Units  Cost_Price  Selling_Price
SKU012  Product 012      Stockout Risk      519.982200           0.0     1256.89        8779.37
SKU034  Product 034      Stockout Risk      356.428788           0.0     1333.43        1518.14
SKU042  Product 042      Stockout Risk      346.732443           0.0     4583.86        9198.83
SKU026  Product 026      Stockout Risk      313.885015           0.0     1799.29       10939.54
SKU018  Product 018      Stockout Risk      283.434982           0.0     5677.05        5575.41
SKU031  Product 031      Stockout Risk      273.864183           0.0     6300.65        8715.93
SKU040  Product 040      Stockout Risk      253.721358           0.0     5169.07        2450.56
SKU010  Product 010      Stockout Risk      235.138295           0.0     3889.06         663.46
SKU017  Product 017      Stockout Risk      219.193953           0.0     6703.26        8477.32
SKU045  Product 045      Stockout Risk  

In [34]:
# Calculate business impact in rupees.

final_risk["Sales_At_Risk"] = (
    final_risk["Shortage_Units"] *
    final_risk["Selling_Price"]
)

final_risk["Capital_Locked_Overstock"] = (
    final_risk["Excess_Units"] *
    final_risk["Cost_Price"]
)

# Total business impact
total_sales_at_risk = final_risk["Sales_At_Risk"].sum()
total_capital_locked = final_risk["Capital_Locked_Overstock"].sum()

print(f"Total Sales at Risk: ₹{total_sales_at_risk:,.2f}")
print(f"Total Capital Locked in Overstock: ₹{total_capital_locked:,.2f}")

Total Sales at Risk: ₹21,066,877.44
Total Capital Locked in Overstock: ₹11,579,868.60


In [35]:
# Rank stockout SKUs by potential sales at risk.

top_sales_risk = (
    final_risk[final_risk["Sales_At_Risk"] > 0]
    .sort_values("Sales_At_Risk", ascending=False)
)

print(
    top_sales_risk[
        [
            "SKU",
            "Product_Name",
            "Shortage_Units",
            "Selling_Price",
            "Sales_At_Risk"
        ]
    ].head(10).to_string(index=False)
)

   SKU Product_Name  Shortage_Units  Selling_Price  Sales_At_Risk
SKU012  Product 012      519.982200        8779.37   4.565116e+06
SKU026  Product 026      313.885015       10939.54   3.433758e+06
SKU042  Product 042      346.732443        9198.83   3.189533e+06
SKU031  Product 031      273.864183        8715.93   2.386981e+06
SKU017  Product 017      219.193953        8477.32   1.858177e+06
SKU018  Product 018      283.434982        5575.41   1.580266e+06
SKU045  Product 045      161.517586        7676.78   1.239935e+06
SKU040  Product 040      253.721358        2450.56   6.217594e+05
SKU034  Product 034      356.428788        1518.14   5.411088e+05
SKU013  Product 013       84.071126        5757.78   4.840630e+05


In [36]:
# Rank overstock SKUs by capital potentially locked.

top_overstock = (
    final_risk[final_risk["Capital_Locked_Overstock"] > 0]
    .sort_values("Capital_Locked_Overstock", ascending=False)
)

print(
    top_overstock[
        [
            "SKU",
            "Product_Name",
            "Excess_Units",
            "Cost_Price",
            "Capital_Locked_Overstock"
        ]
    ].head(10).to_string(index=False)
)

   SKU Product_Name  Excess_Units  Cost_Price  Capital_Locked_Overstock
SKU020  Product 020    566.888844     6700.01              3.798161e+06
SKU006  Product 006    412.107995     6021.91              2.481677e+06
SKU025  Product 025    901.795177     1333.08              1.202165e+06
SKU011  Product 011    697.745462     1718.40              1.199006e+06
SKU036  Product 036    209.213835     5430.12              1.136056e+06
SKU015  Product 015    136.706188     5441.06              7.438266e+05
SKU050  Product 050    519.684390      701.85              3.647405e+05
SKU004  Product 004    188.309398     1445.74              2.722464e+05
SKU038  Product 038     43.034564     4630.58              1.992750e+05
SKU001  Product 001    102.079834     1758.45              1.795023e+05


In [37]:
# Calculate the recommended replenishment quantity
# needed to cover forecast demand plus safety stock.

final_risk["Recommended_Order_Qty"] = (
    final_risk["Cumulative_Forecast"]
    + final_risk["Safety_Stock"]
    - final_risk["Available_Supply"]
).clip(lower=0)

print(
    final_risk[
        [
            "SKU",
            "Risk_Level",
            "Cumulative_Forecast",
            "Available_Supply",
            "Safety_Stock",
            "Recommended_Order_Qty",
            "Recommended_Action"
        ]
    ]
    .sort_values("Recommended_Order_Qty", ascending=False)
    .head(20)
    .to_string(index=False)
)

   SKU         Risk_Level  Cumulative_Forecast  Available_Supply  Safety_Stock  Recommended_Order_Qty           Recommended_Action
SKU012      Stockout Risk           626.982200               107            16             535.982200     Prioritize replenishment
SKU034      Stockout Risk           493.428788               137            34             390.428788     Prioritize replenishment
SKU042      Stockout Risk           556.732443               210            41             387.732443     Prioritize replenishment
SKU018      Stockout Risk           620.434982               337            66             349.434982     Prioritize replenishment
SKU026      Stockout Risk           543.885015               230            31             344.885015     Prioritize replenishment
SKU017      Stockout Risk           340.193953               121           108             327.193953     Prioritize replenishment
SKU031      Stockout Risk           406.864183               133            28     

In [38]:
# Convert recommended order quantity to whole units
# and assign an operational priority.

final_risk["Recommended_Order_Qty"] = np.ceil(
    final_risk["Recommended_Order_Qty"]
).astype(int)

priority_map = {
    "Stockout Risk": 1,
    "Replenishment Risk": 2,
    "Overstock Risk": 3,
    "Healthy": 4
}

final_risk["Priority"] = (
    final_risk["Risk_Level"]
    .map(priority_map)
)

# Create the final planning view
planning_view = (
    final_risk[
        [
            "Priority",
            "SKU",
            "Product_Name",
            "Risk_Level",
            "Cumulative_Forecast",
            "Available_Supply",
            "Projected_Balance",
            "Recommended_Order_Qty",
            "Sales_At_Risk",
            "Capital_Locked_Overstock",
            "Recommended_Action"
        ]
    ]
    .sort_values(
        ["Priority", "Recommended_Order_Qty"],
        ascending=[True, False]
    )
)

print(
    planning_view.head(25).to_string(index=False)
)

 Priority    SKU Product_Name         Risk_Level  Cumulative_Forecast  Available_Supply  Projected_Balance  Recommended_Order_Qty  Sales_At_Risk  Capital_Locked_Overstock           Recommended_Action
        1 SKU012  Product 012      Stockout Risk           626.982200               107        -519.982200                    536   4.565116e+06                       0.0     Prioritize replenishment
        1 SKU034  Product 034      Stockout Risk           493.428788               137        -356.428788                    391   5.411088e+05                       0.0     Prioritize replenishment
        1 SKU042  Product 042      Stockout Risk           556.732443               210        -346.732443                    388   3.189533e+06                       0.0     Prioritize replenishment
        1 SKU018  Product 018      Stockout Risk           620.434982               337        -283.434982                    350   1.580266e+06                       0.0     Prioritize replenishment


In [39]:
# Validate the final SKU-level risk engine.

print("Total SKUs:", final_risk["SKU"].nunique())
print("Total final records:", len(final_risk))
print()

print("Risk distribution:")
print(final_risk["Risk_Level"].value_counts())
print()

print("Negative order quantities:",
      (final_risk["Recommended_Order_Qty"] < 0).sum())

print(
    "Stockout SKUs with zero shortage:",
    (
        (final_risk["Risk_Level"] == "Stockout Risk") &
        (final_risk["Shortage_Units"] <= 0)
    ).sum()
)

print(
    "Healthy SKUs with recommended order:",
    (
        (final_risk["Risk_Level"] == "Healthy") &
        (final_risk["Recommended_Order_Qty"] > 0)
    ).sum()
)

print(
    "Overstock SKUs with recommended order:",
    (
        (final_risk["Risk_Level"] == "Overstock Risk") &
        (final_risk["Recommended_Order_Qty"] > 0)
    ).sum()
)

Total SKUs: 50
Total final records: 50

Risk distribution:
Risk_Level
Stockout Risk         18
Replenishment Risk    13
Overstock Risk        11
Healthy                8
Name: count, dtype: int64

Negative order quantities: 0
Stockout SKUs with zero shortage: 0
Healthy SKUs with recommended order: 0
Overstock SKUs with recommended order: 0


In [40]:
from pathlib import Path

# Create the processed-data directory if it doesn't exist.
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Save forecast results.
forecast_path = PROCESSED_DIR / "forecast_results.csv"
test_results.to_csv(forecast_path, index=False)

# Save final SKU-level risk and planning results.
risk_path = PROCESSED_DIR / "final_risk.csv"
final_risk.to_csv(risk_path, index=False)

print("Saved files:")
print(forecast_path)
print(risk_path)

Saved files:
..\data\processed\forecast_results.csv
..\data\processed\final_risk.csv


In [41]:
# Reload the saved outputs and verify their structure.

forecast_check = pd.read_csv(forecast_path)
risk_check = pd.read_csv(risk_path)

print("Forecast results:")
print("Shape:", forecast_check.shape)
print("Unique SKUs:", forecast_check["SKU"].nunique())
print()

print("Final risk:")
print("Shape:", risk_check.shape)
print("Unique SKUs:", risk_check["SKU"].nunique())
print()

print("Risk distribution:")
print(risk_check["Risk_Level"].value_counts())

Forecast results:
Shape: (600, 4)


Unique SKUs: 50

Final risk:
Shape: (50, 33)
Unique SKUs: 50

Risk distribution:
Risk_Level
Stockout Risk         18
Replenishment Risk    13
Overstock Risk        11
Healthy                8
Name: count, dtype: int64


In [42]:
from pathlib import Path

print("Current working directory:")
print(Path.cwd())

print("\nProject files:")
for path in Path("..").iterdir():
    print(path)

Current working directory:
c:\Users\Vamsi1\Desktop\Zidio\FORESIGHT\notebooks

Project files:
..\dashboard
..\data
..\notebooks
..\outputs
..\reports
..\requirements.txt
..\src


In [43]:
from pathlib import Path

# Go from notebooks/ back to the FORESIGHT project root.
PROJECT_DIR = Path("..")

# Create a dedicated dashboard folder.
DASHBOARD_DIR = PROJECT_DIR / "dashboard"
DASHBOARD_DIR.mkdir(parents=True, exist_ok=True)

print("Dashboard folder:")
print(DASHBOARD_DIR.resolve())

Dashboard folder:
C:\Users\Vamsi1\Desktop\Zidio\FORESIGHT\dashboard


In [44]:
from pathlib import Path

dashboard_code = '''
import streamlit as st
import pandas as pd
from pathlib import Path

# --------------------------------------------------
# Page configuration
# --------------------------------------------------

st.set_page_config(
    page_title="Project FORESIGHT",
    page_icon="📦",
    layout="wide"
)

# --------------------------------------------------
# Load processed risk data
# --------------------------------------------------

DATA_PATH = (
    Path(__file__).resolve().parent
    / "../data/processed/final_risk.csv"
).resolve()

risk_data = pd.read_csv(DATA_PATH)

# --------------------------------------------------
# Dashboard title
# --------------------------------------------------

st.title("📦 Project FORESIGHT")
st.subheader("Inventory Demand & Risk Planning Dashboard")

st.markdown(
    "Use this dashboard to identify stockout risk, "
    "replenishment needs, and excess inventory."
)

# --------------------------------------------------
# Key Performance Indicators
# --------------------------------------------------

total_skus = risk_data["SKU"].nunique()

stockout_skus = (
    risk_data["Risk_Level"] == "Stockout Risk"
).sum()

replenishment_skus = (
    risk_data["Risk_Level"] == "Replenishment Risk"
).sum()

overstock_skus = (
    risk_data["Risk_Level"] == "Overstock Risk"
).sum()

healthy_skus = (
    risk_data["Risk_Level"] == "Healthy"
).sum()

sales_at_risk = risk_data["Sales_At_Risk"].sum()

capital_locked = risk_data[
    "Capital_Locked_Overstock"
].sum()

# --------------------------------------------------
# KPI cards
# --------------------------------------------------

col1, col2, col3, col4 = st.columns(4)

col1.metric("Total SKUs", total_skus)
col2.metric("🔴 Stockout Risk", stockout_skus)
col3.metric("🟠 Replenishment Risk", replenishment_skus)
col4.metric("🔵 Overstock Risk", overstock_skus)

st.divider()

col5, col6 = st.columns(2)

col5.metric(
    "Estimated Sales at Risk",
    f"₹{sales_at_risk:,.0f}"
)

col6.metric(
    "Capital Locked in Overstock",
    f"₹{capital_locked:,.0f}"
)

# --------------------------------------------------
# Risk distribution
# --------------------------------------------------

st.subheader("Risk Distribution")

risk_counts = (
    risk_data["Risk_Level"]
    .value_counts()
    .rename_axis("Risk Level")
    .reset_index(name="SKU Count")
)

st.bar_chart(
    risk_counts.set_index("Risk Level")
)

# --------------------------------------------------
# Quick summary
# --------------------------------------------------

st.subheader("Planning Summary")

st.write(
    f"**{stockout_skus} SKUs** are currently projected "
    "to face stockout risk over the planning horizon."
)

st.write(
    f"**{replenishment_skus} SKUs** require review or "
    "replenishment based on the reorder-point rule."
)

st.write(
    f"**{overstock_skus} SKUs** show potential excess "
    "inventory relative to forecast demand and safety stock."
)
'''

app_path = DASHBOARD_DIR / "app.py"

app_path.write_text(
    dashboard_code,
    encoding="utf-8"
)

print("Dashboard app created:")
print(app_path.resolve())

Dashboard app created:
C:\Users\Vamsi1\Desktop\Zidio\FORESIGHT\dashboard\app.py


In [45]:
# Update the Streamlit dashboard by adding
# the operational "What Do I Reorder?" section.

dashboard_code = '''
import streamlit as st
import pandas as pd
from pathlib import Path

# --------------------------------------------------
# Page configuration
# --------------------------------------------------

st.set_page_config(
    page_title="Project FORESIGHT",
    page_icon="📦",
    layout="wide"
)

# --------------------------------------------------
# Load processed risk data
# --------------------------------------------------

DATA_PATH = (
    Path(__file__).resolve().parent
    / "../data/processed/final_risk.csv"
).resolve()

risk_data = pd.read_csv(DATA_PATH)

# --------------------------------------------------
# Dashboard title
# --------------------------------------------------

st.title("📦 Project FORESIGHT")
st.subheader("Inventory Demand & Risk Planning Dashboard")

st.markdown(
    "Use this dashboard to identify stockout risk, "
    "replenishment needs, and excess inventory."
)

# --------------------------------------------------
# Key metrics
# --------------------------------------------------

total_skus = risk_data["SKU"].nunique()

stockout_skus = (
    risk_data["Risk_Level"] == "Stockout Risk"
).sum()

replenishment_skus = (
    risk_data["Risk_Level"] == "Replenishment Risk"
).sum()

overstock_skus = (
    risk_data["Risk_Level"] == "Overstock Risk"
).sum()

healthy_skus = (
    risk_data["Risk_Level"] == "Healthy"
).sum()

sales_at_risk = risk_data["Sales_At_Risk"].sum()

capital_locked = risk_data[
    "Capital_Locked_Overstock"
].sum()

# --------------------------------------------------
# KPI cards
# --------------------------------------------------

col1, col2, col3, col4 = st.columns(4)

col1.metric("Total SKUs", total_skus)
col2.metric("🔴 Stockout Risk", stockout_skus)
col3.metric("🟠 Replenishment Risk", replenishment_skus)
col4.metric("🔵 Overstock Risk", overstock_skus)

st.divider()

col5, col6 = st.columns(2)

col5.metric(
    "Estimated Sales at Risk",
    f"₹{sales_at_risk:,.0f}"
)

col6.metric(
    "Capital Locked in Overstock",
    f"₹{capital_locked:,.0f}"
)

# --------------------------------------------------
# Risk distribution
# --------------------------------------------------

st.subheader("Risk Distribution")

risk_counts = (
    risk_data["Risk_Level"]
    .value_counts()
    .rename_axis("Risk Level")
    .reset_index(name="SKU Count")
)

st.bar_chart(
    risk_counts.set_index("Risk Level")
)

# --------------------------------------------------
# Planning Summary
# --------------------------------------------------

st.subheader("Planning Summary")

st.write(
    f"**{stockout_skus} SKUs** are currently projected "
    "to face stockout risk over the planning horizon."
)

st.write(
    f"**{replenishment_skus} SKUs** require review or "
    "replenishment based on the reorder-point rule."
)

st.write(
    f"**{overstock_skus} SKUs** show potential excess "
    "inventory relative to forecast demand and safety stock."
)

# --------------------------------------------------
# What Do I Reorder?
# --------------------------------------------------

st.divider()

st.subheader("🚨 What Do I Reorder?")

st.markdown(
    "Priority replenishment list based on projected demand, "
    "available supply, and safety stock."
)

reorder_data = risk_data[
    risk_data["Risk_Level"].isin(
        ["Stockout Risk", "Replenishment Risk"]
    )
].copy()

reorder_data = reorder_data[
    [
        "SKU",
        "Product_Name",
        "Risk_Level",
        "Cumulative_Forecast",
        "Available_Supply",
        "Projected_Balance",
        "Recommended_Order_Qty",
        "Sales_At_Risk",
        "Recommended_Action"
    ]
].sort_values(
    ["Risk_Level", "Recommended_Order_Qty"],
    ascending=[True, False]
)

# Format the table for easier reading
reorder_display = reorder_data.copy()

reorder_display["Cumulative_Forecast"] = (
    reorder_display["Cumulative_Forecast"].round(0).astype(int)
)

reorder_display["Available_Supply"] = (
    reorder_display["Available_Supply"].round(0).astype(int)
)

reorder_display["Projected_Balance"] = (
    reorder_display["Projected_Balance"].round(0).astype(int)
)

reorder_display["Sales_At_Risk"] = (
    reorder_display["Sales_At_Risk"].round(0).astype(int)
)

reorder_display = reorder_display.rename(
    columns={
        "SKU": "SKU",
        "Product_Name": "Product",
        "Risk_Level": "Risk",
        "Cumulative_Forecast": "4-Week Forecast",
        "Available_Supply": "Available Supply",
        "Projected_Balance": "Projected Balance",
        "Recommended_Order_Qty": "Order Qty",
        "Sales_At_Risk": "Sales at Risk",
        "Recommended_Action": "Action"
    }
)

st.dataframe(
    reorder_display,
    use_container_width=True,
    hide_index=True
)

# --------------------------------------------------
# Overstock section
# --------------------------------------------------

st.divider()

st.subheader("📦 Overstock Watchlist")

overstock_data = risk_data[
    risk_data["Risk_Level"] == "Overstock Risk"
].copy()

overstock_data = overstock_data[
    [
        "SKU",
        "Product_Name",
        "Excess_Units",
        "Cost_Price",
        "Capital_Locked_Overstock",
        "Recommended_Action"
    ]
].sort_values(
    "Capital_Locked_Overstock",
    ascending=False
)

overstock_display = overstock_data.copy()

overstock_display["Excess_Units"] = (
    overstock_display["Excess_Units"].round(0).astype(int)
)

overstock_display["Cost_Price"] = (
    overstock_display["Cost_Price"].round(2)
)

overstock_display["Capital_Locked_Overstock"] = (
    overstock_display["Capital_Locked_Overstock"].round(0).astype(int)
)

overstock_display = overstock_display.rename(
    columns={
        "Product_Name": "Product",
        "Excess_Units": "Excess Units",
        "Cost_Price": "Cost Price",
        "Capital_Locked_Overstock": "Capital Locked",
        "Recommended_Action": "Action"
    }
)

st.dataframe(
    overstock_display,
    use_container_width=True,
    hide_index=True
)
'''

app_path.write_text(
    dashboard_code,
    encoding="utf-8"
)

print("Dashboard updated:")
print(app_path.resolve())

Dashboard updated:
C:\Users\Vamsi1\Desktop\Zidio\FORESIGHT\dashboard\app.py


In [46]:
# Fix the reorder table so the most urgent risks
# appear first using the Priority column.

dashboard_code = dashboard_code.replace(
'''reorder_data = reorder_data[
    [
        "SKU",
        "Product_Name",
        "Risk_Level",
        "Cumulative_Forecast",
        "Available_Supply",
        "Projected_Balance",
        "Recommended_Order_Qty",
        "Sales_At_Risk",
        "Recommended_Action"
    ]
].sort_values(
    ["Risk_Level", "Recommended_Order_Qty"],
    ascending=[True, False]
)''',
'''reorder_data = reorder_data[
    [
        "Priority",
        "SKU",
        "Product_Name",
        "Risk_Level",
        "Cumulative_Forecast",
        "Available_Supply",
        "Projected_Balance",
        "Recommended_Order_Qty",
        "Sales_At_Risk",
        "Recommended_Action"
    ]
].sort_values(
    ["Priority", "Recommended_Order_Qty"],
    ascending=[True, False]
)'''
)

dashboard_code = dashboard_code.replace(
'''reorder_display = reorder_display.rename(
    columns={
        "SKU": "SKU",''',
'''reorder_display = reorder_display.rename(
    columns={
        "Priority": "Priority",
        "SKU": "SKU",'''
)

app_path.write_text(
    dashboard_code,
    encoding="utf-8"
)

print("Reorder priority fixed.")

Reorder priority fixed.


In [47]:
# Add an interactive SKU Forecast Explorer
# to the Streamlit dashboard.

dashboard_code = dashboard_code.replace(
'''# --------------------------------------------------
# Overstock section
# --------------------------------------------------

st.divider()''',
'''# --------------------------------------------------
# SKU Forecast Explorer
# --------------------------------------------------

st.divider()

st.subheader("📈 SKU Forecast Explorer")

st.markdown(
    "Compare actual demand with the Random Forest forecast "
    "for the held-out evaluation period."
)

# Load forecast results
FORECAST_PATH = (
    Path(__file__).resolve().parent
    / "../data/processed/forecast_results.csv"
).resolve()

forecast_data = pd.read_csv(FORECAST_PATH)

# Convert date column
forecast_data["Date"] = pd.to_datetime(
    forecast_data["Date"]
)

# SKU selector
selected_sku = st.selectbox(
    "Select SKU",
    sorted(forecast_data["SKU"].unique())
)

# Filter selected SKU
sku_forecast = (
    forecast_data[
        forecast_data["SKU"] == selected_sku
    ]
    .sort_values("Date")
    .copy()
)

# Prepare chart data
forecast_chart = sku_forecast[
    ["Date", "Units_Sold", "Forecast_Units"]
].set_index("Date")

forecast_chart = forecast_chart.rename(
    columns={
        "Units_Sold": "Actual Demand",
        "Forecast_Units": "Forecast"
    }
)

# Display chart
st.line_chart(
    forecast_chart
)

# Display detailed forecast table
st.markdown("**Forecast Details**")

forecast_display = sku_forecast[
    [
        "Date",
        "Units_Sold",
        "Forecast_Units"
    ]
].copy()

forecast_display["Date"] = (
    forecast_display["Date"]
    .dt.strftime("%Y-%m-%d")
)

forecast_display["Units_Sold"] = (
    forecast_display["Units_Sold"]
    .round(0)
    .astype(int)
)

forecast_display["Forecast_Units"] = (
    forecast_display["Forecast_Units"]
    .round(1)
)

forecast_display = forecast_display.rename(
    columns={
        "Date": "Week",
        "Units_Sold": "Actual Demand",
        "Forecast_Units": "Forecast"
    }
)

st.dataframe(
    forecast_display,
    use_container_width=True,
    hide_index=True
)

# --------------------------------------------------
# Overstock section
# --------------------------------------------------

st.divider()'''
)

app_path.write_text(
    dashboard_code,
    encoding="utf-8"
)

print("SKU Forecast Explorer added.")

SKU Forecast Explorer added.


In [48]:
# Add an interactive SKU Planning Detail section
# to the Streamlit dashboard.

dashboard_code = dashboard_code.replace(
'''# --------------------------------------------------
# Overstock section
# --------------------------------------------------

st.divider()

st.subheader("📦 Overstock Watchlist")''',
'''# --------------------------------------------------
# SKU Planning Detail
# --------------------------------------------------

st.divider()

st.subheader("🔎 SKU Planning Detail")

st.markdown(
    "Select a SKU to view its forecast, inventory position, "
    "risk, and recommended action."
)

selected_planning_sku = st.selectbox(
    "Select SKU for planning detail",
    sorted(risk_data["SKU"].unique()),
    key="planning_sku"
)

selected_row = risk_data[
    risk_data["SKU"] == selected_planning_sku
].iloc[0]

# Risk and action
st.markdown(
    f"### {selected_row['Product_Name']} — {selected_row['SKU']}"
)

st.write(
    f"**Risk Level:** {selected_row['Risk_Level']}"
)

st.write(
    f"**Recommended Action:** "
    f"{selected_row['Recommended_Action']}"
)

# Planning metrics
detail_col1, detail_col2, detail_col3 = st.columns(3)

detail_col1.metric(
    "4-Week Forecast",
    f"{selected_row['Cumulative_Forecast']:.0f} units"
)

detail_col2.metric(
    "Available Supply",
    f"{selected_row['Available_Supply']:.0f} units"
)

detail_col3.metric(
    "Projected Balance",
    f"{selected_row['Projected_Balance']:.0f} units"
)

detail_col4, detail_col5, detail_col6 = st.columns(3)

detail_col4.metric(
    "Recommended Order",
    f"{selected_row['Recommended_Order_Qty']:.0f} units"
)

detail_col5.metric(
    "Sales at Risk",
    f"₹{selected_row['Sales_At_Risk']:,.0f}"
)

detail_col6.metric(
    "Capital Locked",
    f"₹{selected_row['Capital_Locked_Overstock']:,.0f}"
)

# --------------------------------------------------
# Overstock section
# --------------------------------------------------

st.divider()

st.subheader("📦 Overstock Watchlist")'''
)

app_path.write_text(
    dashboard_code,
    encoding="utf-8"
)

print("SKU Planning Detail added.")

SKU Planning Detail added.


In [49]:
from pathlib import Path

# Create a dedicated folder for the scoring service.
SERVICE_DIR = PROJECT_DIR / "service"
SERVICE_DIR.mkdir(parents=True, exist_ok=True)

print("Service folder:")
print(SERVICE_DIR.resolve())

Service folder:
C:\Users\Vamsi1\Desktop\Zidio\FORESIGHT\service


In [50]:
# Create the first version of the FORESIGHT scoring API.
# It returns the planning result for a requested SKU.

service_code = '''
from fastapi import FastAPI, HTTPException
import pandas as pd
from pathlib import Path

# --------------------------------------------------
# Create API application
# --------------------------------------------------

app = FastAPI(
    title="Project FORESIGHT Scoring Service",
    description="Returns demand planning and inventory risk results for SKUs.",
    version="1.0.0"
)

# --------------------------------------------------
# Load processed risk data
# --------------------------------------------------

DATA_PATH = (
    Path(__file__).resolve().parent
    / "../data/processed/final_risk.csv"
).resolve()

risk_data = pd.read_csv(DATA_PATH)

# --------------------------------------------------
# Single-SKU scoring endpoint
# --------------------------------------------------

@app.get("/score/{sku}")
def score_sku(sku: str):

    result = risk_data[
        risk_data["SKU"].str.upper() == sku.upper()
    ]

    if result.empty:
        raise HTTPException(
            status_code=404,
            detail=f"SKU '{sku}' not found."
        )

    row = result.iloc[0]

    return {
        "SKU": row["SKU"],
        "Product_Name": row["Product_Name"],
        "Risk_Level": row["Risk_Level"],
        "Forecast_Units_4_Week": round(
            float(row["Cumulative_Forecast"]), 2
        ),
        "Available_Supply": round(
            float(row["Available_Supply"]), 2
        ),
        "Projected_Balance": round(
            float(row["Projected_Balance"]), 2
        ),
        "Recommended_Order_Qty": int(
            round(row["Recommended_Order_Qty"])
        ),
        "Sales_At_Risk": round(
            float(row["Sales_At_Risk"]), 2
        ),
        "Capital_Locked_Overstock": round(
            float(row["Capital_Locked_Overstock"]), 2
        ),
        "Recommended_Action": row["Recommended_Action"]
    }


# --------------------------------------------------
# Health check endpoint
# --------------------------------------------------

@app.get("/")
def health_check():

    return {
        "service": "Project FORESIGHT Scoring Service",
        "status": "running",
        "sku_count": int(risk_data["SKU"].nunique())
    }
'''

service_path = SERVICE_DIR / "app.py"

service_path.write_text(
    service_code,
    encoding="utf-8"
)

print("Scoring service created:")
print(service_path.resolve())

Scoring service created:
C:\Users\Vamsi1\Desktop\Zidio\FORESIGHT\service\app.py


In [51]:
# Add a batch-scoring endpoint to the existing FastAPI service.

batch_endpoint = '''

# --------------------------------------------------
# Batch SKU scoring endpoint
# --------------------------------------------------

@app.get("/score-batch")
def score_batch(skus: str):

    # Convert comma-separated SKU input into a clean list.
    requested_skus = [
        item.strip().upper()
        for item in skus.split(",")
        if item.strip()
    ]

    if not requested_skus:
        raise HTTPException(
            status_code=400,
            detail="Please provide at least one SKU."
        )

    results = []

    for requested_sku in requested_skus:

        match = risk_data[
            risk_data["SKU"].str.upper() == requested_sku
        ]

        if match.empty:
            continue

        row = match.iloc[0]

        results.append({
            "SKU": row["SKU"],
            "Product_Name": row["Product_Name"],
            "Risk_Level": row["Risk_Level"],
            "Forecast_Units_4_Week": round(
                float(row["Cumulative_Forecast"]), 2
            ),
            "Available_Supply": round(
                float(row["Available_Supply"]), 2
            ),
            "Projected_Balance": round(
                float(row["Projected_Balance"]), 2
            ),
            "Recommended_Order_Qty": int(
                round(row["Recommended_Order_Qty"])
            ),
            "Sales_At_Risk": round(
                float(row["Sales_At_Risk"]), 2
            ),
            "Capital_Locked_Overstock": round(
                float(row["Capital_Locked_Overstock"]), 2
            ),
            "Recommended_Action": row["Recommended_Action"]
        })

    return {
        "requested_count": len(requested_skus),
        "matched_count": len(results),
        "results": results
    }
'''

# Insert the batch endpoint before the health-check section.
service_code = service_code.replace(
'''# --------------------------------------------------
# Health check endpoint
# --------------------------------------------------''',
batch_endpoint + '''

# --------------------------------------------------
# Health check endpoint
# --------------------------------------------------'''
)

service_path.write_text(
    service_code,
    encoding="utf-8"
)

print("Batch scoring endpoint added.")

Batch scoring endpoint added.


In [52]:
# Create a minimal requirements.txt specifically for the FORESIGHT API service.

service_requirements = """fastapi
uvicorn
pandas
"""

service_requirements_path = SERVICE_DIR / "requirements.txt"

service_requirements_path.write_text(
    service_requirements,
    encoding="utf-8"
)

print(f"Created: {service_requirements_path.resolve()}")
print("\nContents:")
print(service_requirements)

Created: C:\Users\Vamsi1\Desktop\Zidio\FORESIGHT\service\requirements.txt

Contents:
fastapi
uvicorn
pandas



In [53]:
print((SERVICE_DIR / "requirements.txt").read_text(encoding="utf-8"))

fastapi
uvicorn
pandas

